# BabyROS + Data Engine + FiftyOne

**Transport + contract + curation, in one pipeline.**

This notebook demos a pattern for robotics/physical-AI teams built from three
[Telekinesis](https://docs.telekinesis.ai) / [FiftyOne](https://docs.voxel51.com) pieces that each do one job well:

| Layer | Role | Tool |
|---|---|---|
| **Transport** | move messages between processes/devices, no schema required | [BabyROS](https://github.com/telekinesis-ai/babyros) (pub/sub over Zenoh) |
| **Contract** | give those messages a fixed, typed shape everyone agrees on | Telekinesis [Data Engine](https://docs.telekinesis.ai/data-engine/introduction.html) datatypes (`Category`, `ObjectDetectionAnnotation`, ...) |
| **Curation** | visualize, filter, evaluate, and debug the resulting dataset | [FiftyOne](https://docs.voxel51.com) |

**Story:** a "robot" publishes camera frames + object detections over a BabyROS
topic, using Data Engine's `ObjectDetectionAnnotation` / `Category` types as the
message schema. A BabyROS subscriber ingests those messages into a live FiftyOne
dataset. We then use FiftyOne to do the thing none of the other pieces can do:
find the detector's mistakes.

```
 ┌────────────┐   BabyROS pub/sub    ┌──────────────┐   FiftyOne ingest    ┌───────────────┐
 │  "Robot"   │ ───(Zenoh topic)───▶ │  Subscriber   │ ───(fo.Sample)────▶ │  FiftyOne      │
 │  camera +  │   Data Engine        │  callback     │                    │  dataset       │
 │  detector  │   ObjectDetection    │               │                    │  (curate, eval,│
 └────────────┘   Annotation msgs    └──────────────┘                    │   visualize)   │
                                                                          └───────────────┘
```

### Two data sources, same pipeline
This notebook can run on either:

1. **A real Telekinesis synthetic dataset** — e.g. [Bin Picking: Gear Wheel 20 Teeth](https://docs.telekinesis.ai/data-engine/synthetic-datasets/bin_picking_gear_wheel_20_teeth.html),
   free on [Kaggle](https://www.kaggle.com/telekinesisai). These ship photorealistic
   images + real COCO-format ground truth — but no model predictions, since
   there's no detector bundled with the dataset. Part 2 below synthesizes
   plausible "detector output" by perturbing ground truth (missed detections,
   jittered boxes, noisy confidence) as a stand-in for your real model — swap
   that one function out for an actual inference call and everything downstream
   (BabyROS transport, FiftyOne ingestion, evaluation) is unchanged.
2. **A fully synthetic fallback** (colored circles) if no real dataset is found
   at `DATA_DIR` — so the notebook still runs standalone with zero setup.

### Running this notebook
Written to run two ways:

1. **With the real packages** — `pip install babyros datatypes fiftyone opencv-python numpy` — driving
   an actual Zenoh transport + real Telekinesis datatypes.
2. **Without them** — falls back to a small in-process shim with an *identical
   call signature*, so you can read/run the whole notebook right now, then swap
   in the real packages with zero code changes.

Only `fiftyone` is required to see the curation/debugging section (Part 4) actually execute.


## 0. Setup

Uncomment to install the real stack:

In [ ]:
# %pip install babyros datatypes fiftyone opencv-python-headless numpy


In [ ]:
import glob
import json
import os
import random
import tempfile
import threading

import cv2
import numpy as np

# --------------------------------------------------------------------------
# Try the real Telekinesis / BabyROS packages first. If they aren't
# installed, fall back to a minimal in-process shim that exposes the exact
# same call signatures (babyros.node.Publisher/Subscriber, datatypes.Category,
# datatypes.ObjectDetectionAnnotation). This lets the notebook run anywhere,
# and means swapping in the real packages later requires zero code changes.
# --------------------------------------------------------------------------
try:
    import babyros
    from datatypes import datatypes

    HAVE_BABYROS = True
except ImportError:
    HAVE_BABYROS = False

    class _Category:
        def __init__(self, id, name, supercategory, isthing=1, color=(0, 0, 0)):
            self.id, self.name, self.supercategory = id, name, supercategory
            self.isthing, self.color = isthing, list(color)

        def to_dict(self):
            return dict(
                id=self.id, name=self.name, supercategory=self.supercategory,
                isthing=self.isthing, color=self.color,
            )

    class _ObjectDetectionAnnotation:
        # Mirrors the real datatypes.ObjectDetectionAnnotation contract:
        # required: id, image_id, category_id
        # optional: segmentation, bbox=[x,y,w,h], area, iscrowd, score
        def __init__(self, id, image_id, category_id, bbox=(0, 0, 0, 0),
                     segmentation=None, area=0.0, iscrowd=0, score=None):
            self.id, self.image_id, self.category_id = id, image_id, category_id
            self.bbox = list(bbox)
            self.segmentation = segmentation or []
            self.area, self.iscrowd, self.score = area, iscrowd, score

        def to_dict(self):
            return dict(
                id=self.id, image_id=self.image_id, category_id=self.category_id,
                segmentation=self.segmentation, bbox=self.bbox,
                area=self.area, iscrowd=self.iscrowd, score=self.score,
            )

    class _datatypes_ns:
        Category = _Category
        ObjectDetectionAnnotation = _ObjectDetectionAnnotation

    class _Boxes3D:
        def __init__(self, half_sizes, centers, rotations_in_euler_angle=None):
            self.half_sizes = np.asarray(half_sizes, dtype=np.float32).reshape(-1, 3)
            self.centers = np.asarray(centers, dtype=np.float32).reshape(-1, 3)
            self.rotations_in_euler_angle = (
                np.asarray(rotations_in_euler_angle, dtype=np.float32).reshape(-1, 3)
                if rotations_in_euler_angle is not None else np.zeros_like(self.centers)
            )

        def to_dict(self):
            return dict(
                half_sizes=self.half_sizes.tolist(), centers=self.centers.tolist(),
                rotations_in_euler_angle=self.rotations_in_euler_angle.tolist(),
            )

    class _Points3D:
        def __init__(self, positions, normals=None, colors=None):
            self.positions = np.asarray(positions, dtype=np.float32).reshape(-1, 3)
            self.normals = np.asarray(normals, dtype=np.float32) if normals is not None else None
            self.colors = np.asarray(colors, dtype=np.uint8) if colors is not None else None

        def to_numpy(self, attribute):
            return getattr(self, attribute)

    _datatypes_ns.Boxes3D = _Boxes3D
    _datatypes_ns.Points3D = _Points3D

    datatypes = _datatypes_ns()

    # In-process stand-in for the Zenoh transport underneath BabyROS.
    # Same Publisher/Subscriber/get_topics_in_session() surface as the real
    # `babyros` package (see docs.telekinesis.ai/babyros).
    _BUS = {}
    _BUS_LOCK = threading.Lock()

    class _Publisher:
        def __init__(self, topic):
            self.topic = topic
            with _BUS_LOCK:
                _BUS.setdefault(topic, [])

        def publish(self, data):
            with _BUS_LOCK:
                subs = list(_BUS.get(self.topic, []))
            for cb in subs:
                cb(data)

        def delete(self):
            pass

    class _Subscriber:
        def __init__(self, topic, callback):
            self.topic, self.callback = topic, callback
            with _BUS_LOCK:
                _BUS.setdefault(topic, []).append(callback)

        def delete(self):
            with _BUS_LOCK:
                if self.callback in _BUS.get(self.topic, []):
                    _BUS[self.topic].remove(self.callback)

    class _node_ns:
        Publisher = _Publisher
        Subscriber = _Subscriber

    class _babyros_ns:
        node = _node_ns()

        @staticmethod
        def get_topics_in_session():
            with _BUS_LOCK:
                return list(_BUS.keys())

    babyros = _babyros_ns()

try:
    import fiftyone as fo
    from fiftyone import ViewField as F

    HAVE_FIFTYONE = True
except ImportError:
    HAVE_FIFTYONE = False

print(f"[setup] babyros/datatypes: {'real package' if HAVE_BABYROS else 'in-process shim (same API)'}")
print(f"[setup] fiftyone:          {'installed' if HAVE_FIFTYONE else 'NOT installed — Part 4 will be skipped'}")


## 1. The contract — Data Engine schema

Before anything is sent over the wire, we fix the typed schema both the
publisher (the robot) and the subscriber (the curation layer) agree on. This
is the piece that makes "just send a dict" (BabyROS's whole pitch) safe to do
in practice — everyone fills in the same `Category` / `ObjectDetectionAnnotation`
contract, so the subscriber never has to guess what a message means.

If a real dataset is found below (Part 2), its own COCO `categories` will be
used instead of these placeholders — this fallback only matters in the fully
synthetic path.


In [ ]:
FALLBACK_CATEGORIES = [
    datatypes.Category(id=0, name="red_widget", supercategory="widget", isthing=1, color=(220, 60, 60)),
    datatypes.Category(id=1, name="blue_widget", supercategory="widget", isthing=1, color=(60, 90, 220)),
    datatypes.Category(id=2, name="green_widget", supercategory="widget", isthing=1, color=(60, 180, 90)),
]
FALLBACK_CATEGORY_BY_ID = {c.id: c.to_dict() for c in FALLBACK_CATEGORIES}
FALLBACK_CATEGORY_BY_ID


## 2. The transport — a "robot" publishing over BabyROS

### 2a. Optional: load a real Telekinesis dataset

Point `DATA_DIR` at a downloaded Kaggle export, e.g.

```bash
pip install kaggle              # then set up ~/.kaggle/kaggle.json — see Kaggle account settings
kaggle datasets download -d telekinesisai/bin-picking-gear-wheel-20-teeth -p ./data --unzip
```

These ship as Roboflow-style COCO exports — one `train/`, `valid/`/`val/`, and
`test/` folder, each with flat images plus a single `_annotations.coco.json`.
The loader below reads whichever split it finds first. If `DATA_DIR` doesn't
exist or contains no COCO annotations, we silently fall back to the fully
synthetic generator in 2b — nothing else in the notebook needs to change
either way.


In [ ]:
DATA_DIR = os.path.expanduser("~/data/coco_format")  # <-- point this at your extracted Kaggle dataset


def load_coco_split(split_dir):
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    if not os.path.exists(ann_path):
        candidates = glob.glob(os.path.join(split_dir, "*.json"))
        if not candidates:
            return None
        ann_path = candidates[0]

    with open(ann_path) as f:
        coco = json.load(f)

    categories = {
        c["id"]: datatypes.Category(
            id=c["id"], name=c["name"], supercategory=c.get("supercategory", ""),
            isthing=1, color=tuple(c.get("color", (0, 0, 0))),
        )
        for c in coco.get("categories", [])
    }

    anns_by_image = {}
    for a in coco["annotations"]:
        anns_by_image.setdefault(a["image_id"], []).append(datatypes.ObjectDetectionAnnotation(
            id=a["id"], image_id=a["image_id"], category_id=a["category_id"],
            bbox=a.get("bbox", [0, 0, 0, 0]), segmentation=a.get("segmentation", []),
            area=a.get("area", 0.0), iscrowd=a.get("iscrowd", 0), score=a.get("score"),
        ))

    frames = [
        {
            "image_id": im["id"],
            "filepath": os.path.join(split_dir, im["file_name"]),
            "height": im["height"],
            "width": im["width"],
            "ground_truth": anns_by_image.get(im["id"], []),
        }
        for im in coco["images"]
    ]
    return frames, categories


def find_real_dataset(data_dir):
    for split in ("train", "valid", "val", "test"):
        split_dir = os.path.join(data_dir, split)
        if os.path.isdir(split_dir):
            result = load_coco_split(split_dir)
            if result is not None:
                return result + (split_dir,)
    return None


_real = find_real_dataset(DATA_DIR)
USE_REAL_DATASET = _real is not None

if USE_REAL_DATASET:
    real_frames, real_categories, real_split_dir = _real
    CATEGORY_BY_ID = {cid: c.to_dict() for cid, c in real_categories.items()}
    print(f"[data] found real dataset at {real_split_dir}: {len(real_frames)} images, "
          f"categories={[c['name'] for c in CATEGORY_BY_ID.values()]}")
else:
    CATEGORY_BY_ID = FALLBACK_CATEGORY_BY_ID
    print(f"[data] no real dataset found at '{DATA_DIR}' — using the synthetic generator instead")


### 2b. The "robot": real frames + a simulated detector, or a fully synthetic scene

Either way we end up with, per frame: an image on disk, its Data Engine
`Category` set, ground-truth `ObjectDetectionAnnotation`s, and detector
output. On the real-dataset path there are two interchangeable detector
implementations, selected by `DETECTOR_MODE`:

- `"perturb_gt"` (default) — `simulate_detector_output` perturbs ground
  truth into plausible imperfect predictions: missed detections, jittered
  boxes, noisy confidence, occasional duplicate detections on
  occluded/overlapping instances, and occasional hallucinated boxes on
  background/clutter. Reliable, tunable numbers for a demo.
- `"classical_cv"` — `classical_cv_detector` is a genuinely real detector:
  Hough-circle detection running on the actual image pixels, no ground
  truth involved at all. Weaker and less tunable, but it's the honest
  "plug in a real detector" slot — swap it for an actual model call and
  nothing else in the pipeline changes.

If no real dataset was found, `make_frame`/`toy_detector` render synthetic
colored-circle scenes instead, with the same kinds of injected mistakes
(misses, confused categories, noisy confidence) — `toy_detector` there is
also a real pixel-based detector (color thresholding + contours), just a
simpler one.


In [ ]:
def simulate_detector_output(ground_truth, seed, img_w, img_h,
                              miss_rate=0.10, dup_rate=0.04, hallucination_rate=0.15):
    """Stand-in for 'run your real model here'. The Kaggle datasets ship
    ground truth but no detector output, so this perturbs GT into plausible,
    imperfect predictions:
      - misses (dropped GT boxes)
      - jittered boxes/confidence on the ones it does find
      - duplicate detections on the same instance (double-firing on an
        occluded/overlapping gear -- a real false positive)
      - hallucinated low-confidence boxes on background/clutter with no
        matching GT (also a real false positive)
    This gives FiftyOne's evaluation tools both false negatives *and* false
    positives to find, instead of only misses."""
    rng = random.Random(seed + 999)
    image_id = ground_truth[0].image_id if ground_truth else seed
    predictions, next_id = [], 0

    for g in ground_truth:
        if rng.random() < miss_rate:
            continue
        x, y, w, h = g.bbox
        jw = max(1.0, min(w + rng.uniform(-0.08 * w, 0.08 * w), img_w))
        jh = max(1.0, min(h + rng.uniform(-0.08 * h, 0.08 * h), img_h))
        jx = min(max(0.0, x + rng.uniform(-0.08 * w, 0.08 * w)), img_w - jw)
        jy = min(max(0.0, y + rng.uniform(-0.08 * h, 0.08 * h)), img_h - jh)
        score = max(0.05, min(0.99, rng.gauss(0.85, 0.15)))
        predictions.append(datatypes.ObjectDetectionAnnotation(
            id=next_id, image_id=g.image_id, category_id=g.category_id,
            bbox=[jx, jy, jw, jh], area=jw * jh, iscrowd=0, score=round(score, 3),
        ))
        next_id += 1

        # False positive: double-detect an occluded/overlapping instance
        if rng.random() < dup_rate:
            dw = max(1.0, min(w * rng.uniform(0.7, 1.0), img_w))
            dh = max(1.0, min(h * rng.uniform(0.7, 1.0), img_h))
            dx = min(max(0.0, x + rng.uniform(-0.25 * w, 0.25 * w)), img_w - dw)
            dy = min(max(0.0, y + rng.uniform(-0.25 * h, 0.25 * h)), img_h - dh)
            dscore = max(0.05, min(0.9, rng.gauss(0.55, 0.15)))
            predictions.append(datatypes.ObjectDetectionAnnotation(
                id=next_id, image_id=g.image_id, category_id=g.category_id,
                bbox=[dx, dy, dw, dh], area=dw * dh, iscrowd=0, score=round(dscore, 3),
            ))
            next_id += 1

    # False positive: hallucinate a low-confidence box on background/clutter
    # with no matching ground truth at all
    if rng.random() < hallucination_rate:
        hw = min(rng.uniform(20, 45), img_w)
        hh = min(rng.uniform(20, 45), img_h)
        hx = rng.uniform(0, max(0.0, img_w - hw))
        hy = rng.uniform(0, max(0.0, img_h - hh))
        hscore = max(0.05, min(0.6, rng.gauss(0.35, 0.12)))
        cat_id = rng.choice([g.category_id for g in ground_truth]) if ground_truth else 0
        predictions.append(datatypes.ObjectDetectionAnnotation(
            id=next_id, image_id=image_id, category_id=cat_id,
            bbox=[hx, hy, hw, hh], area=hw * hh, iscrowd=0, score=round(hscore, 3),
        ))
        next_id += 1

    return predictions


def infer_part_category_id(category_by_id):
    """Picks the non-'bin' category -- every one of these Kaggle datasets
    has exactly one foreground 'part' class plus a 'bin' container class."""
    non_bin = [cid for cid, c in category_by_id.items() if c["name"].lower() != "bin"]
    return non_bin[0] if non_bin else next(iter(category_by_id))


def classical_cv_detector(image_path, part_category_id, img_w, img_h, seed):
    """A *real* detector: no ground-truth peeking, runs on the actual pixels.
    This is a classical Hough-circle baseline (gear wheels are roughly
    circular from the top-down camera view) -- not a trained model, but
    genuine inference. Swap this whole function for a YOLO/Detectron2/etc.
    call and nothing else in the pipeline changes: same signature in
    (image path, category id, dims) -> ObjectDetectionAnnotation list out.
    Expect this classical baseline to have modest recall on a cluttered,
    self-occluding pile -- that's realistic, not a bug, and it's exactly the
    kind of gap a trained model (and this whole curation loop) exists to close."""
    img = cv2.imread(image_path)
    if img is None:
        return []
    gray = cv2.medianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 5)
    circles = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=15,
        param1=80, param2=25, minRadius=8, maxRadius=40,
    )
    detections = []
    if circles is not None:
        for i, (cx, cy, r) in enumerate(circles[0]):
            x = max(0.0, min(float(cx - r), img_w - 1))
            y = max(0.0, min(float(cy - r), img_h - 1))
            w = min(float(2 * r), img_w - x)
            h = min(float(2 * r), img_h - y)
            detections.append(datatypes.ObjectDetectionAnnotation(
                id=i, image_id=seed, category_id=part_category_id,
                bbox=[x, y, w, h], area=w * h, iscrowd=0,
                score=0.5,  # Hough has no native confidence -- a real model would report one
            ))
    return detections


# "perturb_gt"   -- reliable, tunable demo numbers (default)
# "classical_cv" -- a genuinely real, pixel-based detector (real dataset only)
DETECTOR_MODE = "perturb_gt"


IMG_SIZE = 256
COLORS_BGR = {0: (60, 60, 220), 1: (220, 90, 60), 2: (90, 180, 60)}  # OpenCV is BGR


def make_frame(seed):
    """Synthetic camera frame + ground-truth boxes, used only when no real
    dataset is found at DATA_DIR."""
    rng = random.Random(seed)
    img = np.full((IMG_SIZE, IMG_SIZE, 3), 30, dtype=np.uint8)
    ground_truth = []
    for ann_id in range(rng.randint(1, 3)):
        cat_id = rng.randint(0, 2)
        r = rng.randint(14, 28)
        cx = rng.randint(r + 2, IMG_SIZE - r - 2)
        cy = rng.randint(r + 2, IMG_SIZE - r - 2)
        cv2.circle(img, (cx, cy), r, COLORS_BGR[cat_id], thickness=-1)
        ground_truth.append(datatypes.ObjectDetectionAnnotation(
            id=ann_id, image_id=seed, category_id=cat_id,
            bbox=[cx - r, cy - r, 2 * r, 2 * r], area=float((2 * r) ** 2),
            iscrowd=0, score=None,
        ))
    return img, ground_truth


def toy_detector(img, seed):
    """A deliberately imperfect 'model' for the synthetic path: color-threshold
    + contour detector with injected misses, confused categories, and noisy
    confidence."""
    rng = random.Random(seed + 999)
    detections, ann_id = [], 0
    for cat_id, bgr in COLORS_BGR.items():
        lower = np.array([max(0, c - 30) for c in bgr], dtype=np.uint8)
        upper = np.array([min(255, c + 30) for c in bgr], dtype=np.uint8)
        mask = cv2.inRange(img, lower, upper)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            if cv2.contourArea(c) < 80:
                continue
            if rng.random() < 0.12:
                continue
            x, y, w, h = cv2.boundingRect(c)
            reported_cat = cat_id
            if rng.random() < 0.08:
                reported_cat = rng.choice([k for k in COLORS_BGR if k != cat_id])
            score = max(0.05, min(0.99, rng.gauss(0.85, 0.18)))
            detections.append(datatypes.ObjectDetectionAnnotation(
                id=ann_id, image_id=seed, category_id=reported_cat,
                bbox=[x, y, w, h], area=float(w * h), iscrowd=0, score=round(score, 3),
            ))
            ann_id += 1
    return detections


Now wire it up over BabyROS: a `Publisher` on topic `robot/perception` sends
one envelope per frame — image path + Data Engine categories + ground truth +
detections — and a `Subscriber` on the same topic collects everything into
`received_envelopes`. This is the only part of the notebook that touches
BabyROS directly; note it never needed to know what an `ObjectDetectionAnnotation`
*means* — only the publisher and the eventual consumer (FiftyOne, below) do.


In [ ]:
FRAMES_DIR = tempfile.mkdtemp(prefix="babyros_fo_demo_")
received_envelopes = []


def on_perception_msg(msg):
    """BabyROS subscriber callback -- just appends the raw envelope. In a
    real deployment you'd hand this off to a queue/writer thread instead of
    blocking the Zenoh callback."""
    received_envelopes.append(msg)


perception_pub = babyros.node.Publisher(topic="robot/perception")
perception_sub = babyros.node.Subscriber(topic="robot/perception", callback=on_perception_msg)
print("Active topics in session:", babyros.get_topics_in_session())

N_FRAMES = 40

if USE_REAL_DATASET:
    part_category_id = infer_part_category_id(CATEGORY_BY_ID)
    for frame in real_frames[:N_FRAMES]:
        if DETECTOR_MODE == "classical_cv":
            detections = classical_cv_detector(
                frame["filepath"], part_category_id,
                img_w=frame["width"], img_h=frame["height"], seed=frame["image_id"],
            )
        else:
            detections = simulate_detector_output(
                frame["ground_truth"], seed=frame["image_id"],
                img_w=frame["width"], img_h=frame["height"],
            )
        envelope = {
            "image_id": frame["image_id"],
            "filepath": frame["filepath"],
            "height": frame["height"],
            "width": frame["width"],
            "categories": CATEGORY_BY_ID,
            "ground_truth": [g.to_dict() for g in frame["ground_truth"]],
            "detections": [d.to_dict() for d in detections],
        }
        perception_pub.publish(data=envelope)
else:
    for frame_id in range(N_FRAMES):
        img, ground_truth = make_frame(seed=frame_id)
        detections = toy_detector(img, seed=frame_id)

        frame_path = os.path.join(FRAMES_DIR, f"frame_{frame_id:04d}.png")
        cv2.imwrite(frame_path, img)

        envelope = {
            "image_id": frame_id,
            "filepath": frame_path,
            "height": img.shape[0],
            "width": img.shape[1],
            "categories": CATEGORY_BY_ID,
            "ground_truth": [g.to_dict() for g in ground_truth],
            "detections": [d.to_dict() for d in detections],
        }
        perception_pub.publish(data=envelope)

perception_pub.delete()
perception_sub.delete()
print(f"Ingested {len(received_envelopes)} frames over BabyROS topic 'robot/perception' "
      f"({'real dataset, detector=' + DETECTOR_MODE if USE_REAL_DATASET else 'synthetic'})")


## 3. The curation layer — loading into FiftyOne

Each envelope's `detections` / `ground_truth` are plain dicts shaped exactly
like `ObjectDetectionAnnotation.to_dict()` (COCO-style `bbox=[x,y,w,h]` in
pixels). FiftyOne's `Detection.bounding_box` wants `[x,y,w,h]` normalized to
`[0,1]`, so the only translation needed is a divide by image width/height —
everything else about the schema carries over directly, because the contract
was fixed up front. This cell doesn't care whether the data came from the
real dataset or the synthetic generator.


In [ ]:
def annotations_to_fo_detections(anns, categories_by_id, img_w, img_h):
    dets = []
    for a in anns:
        x, y, w, h = a["bbox"]
        dets.append(fo.Detection(
            label=categories_by_id[a["category_id"]]["name"],
            bounding_box=[x / img_w, y / img_h, w / img_w, h / img_h],
            confidence=a.get("score"),
        ))
    return fo.Detections(detections=dets)


if HAVE_FIFTYONE:
    dataset_name = "babyros_perception_demo"
    if dataset_name in fo.list_datasets():
        fo.delete_dataset(dataset_name)
    dataset = fo.Dataset(dataset_name, persistent=False)

    samples = []
    for env in received_envelopes:
        sample = fo.Sample(filepath=env["filepath"])
        sample["frame_id"] = env["image_id"]
        sample["predictions"] = annotations_to_fo_detections(
            env["detections"], env["categories"], env["width"], env["height"]
        )
        sample["ground_truth"] = annotations_to_fo_detections(
            env["ground_truth"], env["categories"], env["width"], env["height"]
        )
        samples.append(sample)

    dataset.add_samples(samples)
    print(dataset)
else:
    print("Install fiftyone (`pip install fiftyone`) to run this cell and Part 4 below.")


## 4. Curate & debug

This is the part BabyROS and the Data Engine schema can't do for you:
finding *which* frames are wrong, and why.


In [ ]:
if HAVE_FIFTYONE:
    # Frames where the detector reported at least one low-confidence box
    low_conf_view = dataset.filter_labels("predictions", F("confidence") < 0.5, only_matches=True)
    print(f"{len(low_conf_view)} / {len(dataset)} frames have a low-confidence detection")

    # Detections per category (spot class imbalance or a category the
    # detector systematically confuses)
    print("Detections per category:", dataset.count_values("predictions.detections.label"))

    # Frames with zero detections at all (candidate missed frames)
    empty_view = dataset.match(F("predictions.detections").length() == 0)
    empty_view.tag_samples("no_detections")
    print(f"Tagged {len(empty_view)} frames with zero detections")


Because both `predictions` and `ground_truth` are built from the same
`ObjectDetectionAnnotation` contract, FiftyOne can score the detector
directly against ground truth — matching boxes, computing precision/recall
per class, and tagging every individual detection as a true positive, false
positive, or false negative.


In [ ]:
if HAVE_FIFTYONE:
    results = dataset.evaluate_detections(
        "predictions", gt_field="ground_truth", eval_key="eval", compute_mAP=True,
    )
    print(f"mAP: {results.mAP():.3f}\n")
    results.print_report()

    fp_view = dataset.filter_labels("predictions", F("eval") == "fp", only_matches=True)
    fn_view = dataset.filter_labels("ground_truth", F("eval") == "fn", only_matches=True)
    print(f"\n{len(fp_view)} frames contain a false positive")
    print(f"{len(fn_view)} frames contain a missed (false negative) detection")


### Saved views (2D)

Views are computed on demand; **saved views** persist a named view on the
dataset itself, so anyone opening the App later can jump straight to
"show me the false positives" from the view bar's dropdown instead of
re-typing the filter.


In [ ]:
if HAVE_FIFTYONE:
    for name, view in [
        ("false_positives", fp_view),
        ("false_negatives", fn_view),
        ("low_confidence_predictions", low_conf_view),
        ("no_detections", dataset.match_tags("no_detections")),
    ]:
        if name in dataset.list_saved_views():
            dataset.delete_saved_view(name)
        dataset.save_view(name, view)

    print("Saved views:", dataset.list_saved_views())
    print("Open the App's view bar -> the bookmark/dropdown icon -> pick one of the names above.")


Optional: embed and visualize the dataset in 2D (skipped automatically if a backing model/embedding isn't available).

In [ ]:
if HAVE_FIFTYONE:
    try:
        import fiftyone.brain as fob

        fob.compute_visualization(dataset, brain_key="perception_viz")
        print("Computed a 2D embeddings visualization — open it in the App's Embeddings panel.")
    except Exception as e:
        print(f"Skipped embeddings visualization ({e}). This step needs extra deps (e.g. torch).")


In [ ]:
if HAVE_FIFTYONE:
    session = fo.launch_app(dataset)
    session
else:
    print("session = fo.launch_app(dataset)  # once fiftyone is installed")


## 6. Go 3D — the same pattern with `Boxes3D` / `Points3D`

Everything so far has been 2D (`ObjectDetectionAnnotation`, `Boxes2D`-shaped
messages). The pattern is identical for 3D perception — a point cloud plus
`Boxes3D` ground truth/predictions over a different BabyROS topic — using
Data Engine's actual `Points3D` (positions/normals/colors) and `Boxes3D`
(centers/half-extents/rotations) types.

This section builds a standalone synthetic 3D bin-picking scene (no Kaggle
dependency — those datasets ship 2D COCO annotations, not 3D), since it's
demonstrating the *pattern*, not another real dataset. **One caveat:** the
FiftyOne 3D App API (`fo.Scene`, `fo.PointCloud`,
`Detection(location=..., dimensions=..., rotation=...)`) here is built from
FiftyOne's documented 3D visualizer conventions, not verified against every
installed version. If your version's API differs, the transport/schema/
perturbation code above the FiftyOne-specific cell is unaffected either
way; only the last cell (building the `.fo3d` scene + loading it into a
dataset) might need a small adjustment. Check https://docs.voxel51.com if so.


In [ ]:
import math

BIN_W, BIN_D, BIN_H = 0.30, 0.20, 0.10  # meters -- a small tray, roughly


def make_scene_3d(seed, n_objects=None):
    """Synthetic 3D bin-picking scene: N small discs (standing in for gear
    wheels) settled at random positions/tilts inside a tray footprint."""
    rng = random.Random(seed)
    n_objects = n_objects or rng.randint(8, 14)
    centers, half_sizes, rotations = [], [], []
    for _ in range(n_objects):
        r = rng.uniform(0.015, 0.025)
        t = rng.uniform(0.006, 0.012)
        cx = rng.uniform(-BIN_W / 2 + r, BIN_W / 2 - r)
        cy = rng.uniform(-BIN_D / 2 + r, BIN_D / 2 - r)
        cz = rng.uniform(t, BIN_H * 0.6)
        centers.append([cx, cy, cz])
        half_sizes.append([r, r, t / 2])
        rotations.append([rng.uniform(-0.3, 0.3), rng.uniform(-0.3, 0.3), rng.uniform(0, math.pi)])
    return datatypes.Boxes3D(half_sizes=half_sizes, centers=centers, rotations_in_euler_angle=rotations)


def sample_point_cloud(boxes3d, seed, floor_points=800, points_per_object=60):
    """A crude stand-in for a depth-camera point cloud: a noisy floor plane
    plus a ring of points on each object's top surface."""
    rng = np.random.RandomState(seed)
    pts = [np.stack([
        rng.uniform(-BIN_W / 2, BIN_W / 2, floor_points),
        rng.uniform(-BIN_D / 2, BIN_D / 2, floor_points),
        rng.normal(0, 0.001, floor_points),
    ], axis=1)]
    for c, hs in zip(boxes3d.centers, boxes3d.half_sizes):
        theta = rng.uniform(0, 2 * math.pi, points_per_object)
        rr = rng.uniform(0, hs[0], points_per_object)
        pts.append(np.stack([
            c[0] + rr * np.cos(theta),
            c[1] + rr * np.sin(theta),
            np.full(points_per_object, c[2] + hs[2]) + rng.normal(0, 0.0008, points_per_object),
        ], axis=1))
    return datatypes.Points3D(positions=np.concatenate(pts, axis=0).astype(np.float32))


def perturb_boxes3d(boxes3d, seed, miss_rate=0.10, dup_rate=0.05, hallucination_rate=0.15):
    """Same idea as simulate_detector_output, for 3D boxes: misses, jitter,
    occasional duplicates, occasional hallucinations."""
    rng = random.Random(seed + 999)
    centers, half_sizes, rotations, scores = [], [], [], []
    for c, hs, rot in zip(boxes3d.centers, boxes3d.half_sizes, boxes3d.rotations_in_euler_angle):
        if rng.random() < miss_rate:
            continue
        jc = [float(c[i] + rng.uniform(-0.1, 0.1) * hs[i]) for i in range(3)]
        jhs = [max(0.002, float(hs[i] + rng.uniform(-0.1, 0.1) * hs[i])) for i in range(3)]
        jrot = [float(rot[i] + rng.uniform(-0.15, 0.15)) for i in range(3)]
        centers.append(jc); half_sizes.append(jhs); rotations.append(jrot)
        scores.append(max(0.05, min(0.99, rng.gauss(0.85, 0.15))))
        if rng.random() < dup_rate:
            centers.append(jc); half_sizes.append(jhs); rotations.append(jrot)
            scores.append(max(0.05, min(0.9, rng.gauss(0.5, 0.15))))
    if rng.random() < hallucination_rate and len(boxes3d.centers) > 0:
        base = boxes3d.centers[rng.randrange(len(boxes3d.centers))]
        centers.append([float(base[0] + rng.uniform(-0.05, 0.05)),
                         float(base[1] + rng.uniform(-0.05, 0.05)),
                         float(rng.uniform(0.01, BIN_H))])
        half_sizes.append([0.02, 0.02, 0.005])
        rotations.append([0.0, 0.0, rng.uniform(0, math.pi)])
        scores.append(max(0.05, min(0.6, rng.gauss(0.3, 0.1))))
    boxes = (
        datatypes.Boxes3D(half_sizes=half_sizes, centers=centers, rotations_in_euler_angle=rotations)
        if centers else None
    )
    return boxes, scores


def write_pcd(path, positions):
    """Minimal ASCII .pcd writer (no open3d/pypcd dependency)."""
    n = positions.shape[0]
    with open(path, "w") as f:
        f.write("# .PCD v0.7 - Point Cloud Data file format\n")
        f.write("VERSION 0.7\nFIELDS x y z\nSIZE 4 4 4\nTYPE F F F\nCOUNT 1 1 1\n")
        f.write(f"WIDTH {n}\nHEIGHT 1\nVIEWPOINT 0 0 0 1 0 0 0\nPOINTS {n}\nDATA ascii\n")
        for p in positions:
            f.write(f"{p[0]:.6f} {p[1]:.6f} {p[2]:.6f}\n")


Wire it up over BabyROS exactly like the 2D case, on a separate topic:

In [ ]:
SCENE_3D_DIR = tempfile.mkdtemp(prefix="babyros_fo_demo_3d_")
received_envelopes_3d = []


def on_perception_3d_msg(msg):
    received_envelopes_3d.append(msg)


perception_3d_pub = babyros.node.Publisher(topic="robot/perception_3d")
perception_3d_sub = babyros.node.Subscriber(topic="robot/perception_3d", callback=on_perception_3d_msg)

N_FRAMES_3D = 10
for frame_id in range(N_FRAMES_3D):
    gt_boxes = make_scene_3d(seed=frame_id)
    cloud = sample_point_cloud(gt_boxes, seed=frame_id)
    pred_boxes, pred_scores = perturb_boxes3d(gt_boxes, seed=frame_id)

    pcd_path = os.path.join(SCENE_3D_DIR, f"scene_{frame_id:04d}.pcd")
    write_pcd(pcd_path, cloud.to_numpy("positions"))

    envelope = {
        "frame_id": frame_id,
        "pcd_path": pcd_path,
        "ground_truth_3d": gt_boxes.to_dict(),
        "predictions_3d": pred_boxes.to_dict() if pred_boxes is not None else None,
        "predictions_3d_scores": pred_scores,
    }
    perception_3d_pub.publish(data=envelope)

perception_3d_pub.delete()
perception_3d_sub.delete()
print(f"Ingested {len(received_envelopes_3d)} 3D scenes over BabyROS topic 'robot/perception_3d'")


Load into FiftyOne as 3D samples. Each `.fo3d` scene file references one
point cloud; 3D detections use `location`/`dimensions`/`rotation` instead of
`bounding_box`. This is the one cell in the notebook that hasn't been
verified against a live FiftyOne install (see the caveat above) -- everything
above it (BabyROS transport, Data Engine `Boxes3D`/`Points3D` schema,
perturbation) is independent of FiftyOne and already verified.


In [ ]:
def boxes3d_dict_to_fo_detections(boxes3d_dict, label, scores=None):
    dets = []
    centers = boxes3d_dict["centers"]
    half_sizes = boxes3d_dict["half_sizes"]
    rotations = boxes3d_dict.get("rotations_in_euler_angle") or [[0, 0, 0]] * len(centers)
    for i, (c, hs, rot) in enumerate(zip(centers, half_sizes, rotations)):
        dets.append(fo.Detection(
            label=label,
            location=list(c),
            dimensions=[2 * hs[0], 2 * hs[1], 2 * hs[2]],
            rotation=list(rot),
            confidence=(scores[i] if scores else None),
        ))
    return fo.Detections(detections=dets)


if HAVE_FIFTYONE:
    try:
        dataset_3d_name = "babyros_perception_demo_3d"
        if dataset_3d_name in fo.list_datasets():
            fo.delete_dataset(dataset_3d_name)
        dataset_3d = fo.Dataset(dataset_3d_name, persistent=False)

        samples_3d = []
        for env in received_envelopes_3d:
            scene = fo.Scene()
            scene.add(fo.PointCloud(name="points", pcd_path=env["pcd_path"]))
            scene_path = env["pcd_path"].replace(".pcd", ".fo3d")
            scene.write(scene_path)

            sample = fo.Sample(filepath=scene_path)
            sample["frame_id"] = env["frame_id"]
            sample["ground_truth"] = boxes3d_dict_to_fo_detections(env["ground_truth_3d"], "gear")
            if env["predictions_3d"] is not None:
                sample["predictions"] = boxes3d_dict_to_fo_detections(
                    env["predictions_3d"], "gear", env["predictions_3d_scores"]
                )
            samples_3d.append(sample)

        dataset_3d.add_samples(samples_3d)
        print(dataset_3d)
        session_3d = fo.launch_app(dataset_3d)
    except Exception as e:
        print(f"Couldn't build the 3D FiftyOne dataset ({e}).")
        print("The transport/schema/perturbation code above is unaffected -- "
              "check https://docs.voxel51.com for your installed version's 3D API.")
else:
    print("Install fiftyone to run this cell.")


### Saved views (3D)

No `evaluate_detections` call was run on `dataset_3d` above (FiftyOne's 3D
IoU-based evaluation API is another surface that hasn't been verified against
a live install -- see the caveat earlier in this section), so these views
compare raw detection counts instead of true/false positive tags. Still
genuinely useful: a scene with fewer predicted boxes than ground-truth boxes
is a stand-in for "the detector probably missed something in this pile."


In [ ]:
if HAVE_FIFTYONE:
    try:
        fewer_preds_view = dataset_3d.match(
            F("predictions.detections").length() < F("ground_truth.detections").length()
        )
        has_preds_view = dataset_3d.match(F("predictions.detections").length() > 0)

        for name, view in [
            ("fewer_predictions_than_ground_truth", fewer_preds_view),
            ("has_predictions", has_preds_view),
        ]:
            if name in dataset_3d.list_saved_views():
                dataset_3d.delete_saved_view(name)
            dataset_3d.save_view(name, view)

        print("Saved views (3D):", dataset_3d.list_saved_views())
    except Exception as e:
        print(f"Couldn't create 3D saved views ({e}). "
              "The underlying dataset/scenes above are unaffected.")


## 7. Go live — dataset grows while the "robot" runs

Everything so far batch-collects all envelopes, *then* loads them into
FiftyOne. For a live demo, the subscriber callback should write directly
into the dataset as messages arrive, and the publisher should run on its own
thread instead of firing every message instantly.

This version runs **indefinitely** -- one new sample added every 10 seconds
-- until you run the explicit stop cell further down. It builds its own
dataset/topic so it doesn't disturb `dataset` from Part 3.


In [ ]:
import itertools
import threading
import time

if HAVE_FIFTYONE:
    live_dataset_name = "babyros_perception_demo_live"
    if live_dataset_name in fo.list_datasets():
        fo.delete_dataset(live_dataset_name)
    live_dataset = fo.Dataset(live_dataset_name, persistent=False)
    _live_lock = threading.Lock()

    def on_live_msg(msg):
        """Runs on the BabyROS callback thread -- ingest straight into the
        live dataset instead of buffering. add_sample() isn't guaranteed
        thread-safe across all FiftyOne backends, so we serialize writes
        with a lock; reads (e.g. len(dataset), the App) are safe concurrently."""
        sample = fo.Sample(filepath=msg["filepath"])
        sample["frame_id"] = msg["image_id"]
        sample["predictions"] = annotations_to_fo_detections(
            msg["detections"], msg["categories"], msg["width"], msg["height"]
        )
        sample["ground_truth"] = annotations_to_fo_detections(
            msg["ground_truth"], msg["categories"], msg["width"], msg["height"]
        )
        with _live_lock:
            live_dataset.add_sample(sample)

    live_pub = babyros.node.Publisher(topic="robot/perception_live")
    live_sub = babyros.node.Subscriber(topic="robot/perception_live", callback=on_live_msg)

    LIVE_INTERVAL_S = 10.0  # seconds between each new sample
    _live_stop_event = threading.Event()

    def _make_live_envelope(item):
        if USE_REAL_DATASET:
            frame = item
            dets = (classical_cv_detector(frame["filepath"], part_category_id,
                                           frame["width"], frame["height"], frame["image_id"])
                    if DETECTOR_MODE == "classical_cv" else
                    simulate_detector_output(frame["ground_truth"], frame["image_id"],
                                              frame["width"], frame["height"]))
            return {
                "image_id": frame["image_id"], "filepath": frame["filepath"],
                "height": frame["height"], "width": frame["width"],
                "categories": CATEGORY_BY_ID,
                "ground_truth": [g.to_dict() for g in frame["ground_truth"]],
                "detections": [d.to_dict() for d in dets],
            }
        else:
            frame_id = item
            img, gt = make_frame(seed=frame_id)
            dets = toy_detector(img, seed=frame_id)
            frame_path = os.path.join(FRAMES_DIR, f"live_frame_{frame_id:04d}.png")
            cv2.imwrite(frame_path, img)
            return {
                "image_id": frame_id, "filepath": frame_path,
                "height": img.shape[0], "width": img.shape[1],
                "categories": CATEGORY_BY_ID,
                "ground_truth": [g.to_dict() for g in gt],
                "detections": [d.to_dict() for d in dets],
            }

    def _publish_live():
        # Real dataset: cycle through it repeatedly if the demo runs longer
        # than the dataset is long. Synthetic: infinite increasing seeds.
        source = (
            itertools.cycle(real_frames) if (USE_REAL_DATASET and real_frames)
            else itertools.count()
        )
        count = 0
        for item in source:
            if _live_stop_event.is_set():
                break
            envelope = _make_live_envelope(item)
            live_pub.publish(data=envelope)
            count += 1
            print(f"[live publisher] added sample #{count} -- "
                  f"live_dataset now has {len(live_dataset)} samples")
            # Sleep in small increments so a stop request lands within ~0.2s
            # instead of waiting out the full interval.
            for _ in range(int(LIVE_INTERVAL_S / 0.2)):
                if _live_stop_event.is_set():
                    break
                time.sleep(0.2)
        print("[live publisher] stopped.")

    live_session = fo.launch_app(live_dataset)
    print(f"App launched on an empty dataset -- adding a new sample every "
          f"{LIVE_INTERVAL_S:.0f}s. Run the STOP cell further down to end this.")
    _publisher_thread = threading.Thread(target=_publish_live, daemon=True)
    _publisher_thread.start()
else:
    print("Install fiftyone to run this cell.")


Run this any time to check progress -- growth should be visible within one
`LIVE_INTERVAL_S` window regardless of when you run it, since the publisher
keeps going until you explicitly stop it below.


In [ ]:
if HAVE_FIFTYONE:
    print(f"live_dataset currently has {len(live_dataset)} samples")


**Stop cell** -- run this to end the live stream:

In [ ]:
if HAVE_FIFTYONE:
    _live_stop_event.set()
    _publisher_thread.join(timeout=LIVE_INTERVAL_S + 5)
    live_pub.delete()
    live_sub.delete()
    print(f"Stopped. live_dataset ended with {len(live_dataset)} samples.")


## 8. Where this goes further

- **A real trained detector**: `classical_cv_detector` (Part 2) is the
  honest "plug in a real detector" skeleton -- same signature, swap the
  body for a YOLO/Detectron2/etc. inference call.
- **Try the other Kaggle datasets**: Hinge 09, Rod End 01, Pistons,
  Flange 4 Holes, Bracket 01/02 all use the same COCO export layout, so
  just re-point `DATA_DIR`.
- **Real 3D data**: Part 6 uses a synthetic point cloud since the Kaggle
  exports are 2D-only; a real depth camera or LIDAR publishing `Points3D`
  over BabyROS drops into the same pipeline.
- **A real live robot**: Part 7's publisher thread is standing in for an
  actual sensor loop -- point `on_live_msg` at a real BabyROS topic from a
  running robot and the ingestion side is unchanged.
